# Lektion 09 - Metakognitiv designmönster


## Setup

Denna anteckningsbok demonstrerar Metakognitionsdesignmönstret med Microsoft Agent Framework.

**Förutsättningar:**
- Azure OpenAI-distribution konfigurerad via miljövariabler
- Azure CLI autentiserad (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Vad är Metakognition?

Metakognition är **att tänka på tänkande**. I samband med AI-agenter betyder det att bygga agenter som kan:

- **Självreflektera** över sina egna resultat och resonemangsprocess
- **Upptäcka fel** och återhämta sig smidigt istället för att tyst misslyckas
- **Utvärdera** om deras svar är fullständiga och hjälpsamma
- **Anpassa** sin strategi när en initial metod inte fungerar (t.ex. använda ett reservsystem)

En metakognitiv agent svarar inte bara på frågor — den övervakar sin egen prestation och justerar i farten.


## Primära och Reservverktyg

Ett vanligt metakognitionsmönster är **fallback-strategin**. Agenten försöker först ett primärt verktyg; om det misslyckas (t.ex. ett 404-fel), känner agenten igen felet och byter transparent till ett reservverktyg.

Detta speglar verkliga system där primära tjänster kan vara otillgängliga och agenter måste självdiagnostisera problemet innan de väljer en alternativ väg.

Nedan definierar vi två sökverktyg för flyg:
- **Primärt** — täcker Paris, Tokyo och Barcelona
- **Reserv** — täcker Berlin, Sydney och New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Självreflekterande agent med felåterställning

Agenten nedan får instruktioner att först försöka använda det primära flygsystemet, känna igen fel och transparent falla tillbaka på backupsystemet. Efter varje svar utvärderar den kort om den fullständigt besvarade användarens fråga.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mönster för Självutvärdering

En annan aspekt av metakognition är **självutvärdering**: en separat agent (eller samma agent vid ett andra genomgång) granskar ett svar för fullständighet, noggrannhet och hjälpsamhet.

Nedan skapar vi en `ResponseEvaluator`-agent som betygsätter reseagenters svar på tre dimensioner.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sammanfattning

I denna lektion lärde du dig hur man bygger **metakognitiva agenter** med Microsoft Agent Framework:

- **Självreflektion**: Agenter som övervakar sitt eget resonerande och transparent kommunicerar vad som hände.
- **Felfunktion med reservlösningar**: Ett primärt + reservverktygsmönster där agenten upptäcker fel (t.ex. 404-fel) och automatiskt försöker med en alternativ källa.
- **Självutvärdering**: En separat utvärderingsagent som bedömer svar för fullständighet, noggrannhet och hjälpsamhet.

Dessa mönster gör agenter mer robusta, transparenta och pålitliga — viktiga egenskaper för produktionsanvändning.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
